# 07 · Planta por hora (Maui)

CAMMESA publica solo datos mensuales. Para saber cuánto se gana con datos por hora se usa una planta pública: Maui Ocean Center, NREL PVDAQ 2105. Corre con los crudos de `data/raw/maui` (`python -m src.maui` los baja si faltan).

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import DATA_PROCESSED
from src import maui
M = json.loads((DATA_PROCESSED / 'maui_evidencia.json').read_text())
{k: M[k] for k in ('planta', 'fuente', 'desde', 'hasta', 'potencia_kw')}

## Primero, mirar los datos

El inversor 11 es una copia del 04. Se empareja cada lectura con la del otro equipo y se cuentan las idénticas. Otro par sirve de control.

In [ ]:
q = M['calidad']; display(pd.DataFrame({'inv04 e inv11': q['inv11_copia_de_inv04'], 'inv04 e inv10': q['control_inv04_inv10']}).round(4)); q['inv06_kw_max'], q['sensor']

## La regla

Cada inversor contra la mediana de sus vecinos a la misma hora, corregido por su forma habitual (2019-2021). Dos horas de sol por debajo del 20 % o cuatro por debajo del 80 % dan aviso. No usa satélite ni modelo: todos ven el mismo cielo.

In [ ]:
m = M['mapa']; z = np.array([[np.nan if v is None else v for v in f] for f in m['z']], float)
plt.figure(figsize=(11, 3.5)); plt.imshow(z, aspect='auto', vmin=.4, vmax=1.1, cmap='Blues_r'); plt.yticks(range(len(m['inversores'])), m['inversores'])
plt.xlabel('semana'); plt.colorbar(label='rinde');

In [ ]:
pd.DataFrame(M['episodios']).round(2)

## Cuánto tarda en avisar

Se inyectan fallas sobre inversores sanos de 2022-2023 y se cuentan horas de sol. Las fallas inyectadas solo miden: no entrenan nada, y las causas reales no están verificadas en la planta.

In [ ]:
filas = [dict(planta='Maui, Hawái', equipos=len(m['inversores']), **{k: v['mediana_horas'] for k, v in M['detector'].items()}, cae15_detecta=M['detector']['parcial_15']['detectadas'], avisos_mes=M['avisos_sin_falla']['por_inversor_mes'])]
for o in M.get('otras_plantas', []):
    filas.append(dict(planta=o['planta'], equipos=o['equipos'], **{k: v['mediana_horas'] for k, v in o['detector'].items()}, cae15_detecta=o['detector']['parcial_15']['detectadas'], avisos_mes=o['avisos_sin_falla']['por_inversor_mes']))
pd.DataFrame(filas).round(2)

**Lectura.** Con los mismos umbrales, en tres plantas muy distintas, una falla grande se avisa en horas de sol. Una caída de 15 % se ve poco y tarda días: es el límite de la regla. Una suciedad pareja en todos los equipos tampoco se ve: al comparar con los vecinos, se cancela.

In [ ]:
print(f"Energía faltante en caídas largas: {M['faltante_kwh']/1000:.1f} MWh, {M['faltante_kwh']/M['energia_kwh']:.1%} de lo generado.")